In [30]:
# Core
import json
from pathlib import Path
from typing import Dict, Tuple, List

# Numeric
import numpy as np
import pandas as pd

# Medical imaging
import nibabel as nib

# Instance analysis
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

# Visualization (optional but recommended)
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 200)


In [31]:
def load_summary_json(path: Path) -> Dict:
    with open(path, "r") as f:
        return json.load(f)

summary_a = load_summary_json(Path("/Users/ossner/git/VoronoiLoss/data/nnUNet_predictions/Dataset501_BrainMets/nnUNetTrainerDiceCEBaseline__nnUNetPlans__3d_fullres/experiment_00/fold_0/summary.json"))
summary_b = load_summary_json(Path("/Users/ossner/git/VoronoiLoss/data/nnUNet_predictions/Dataset501_BrainMets/nnUNetTrainerGlobalCCDiceCE__nnUNetPlans__3d_fullres/experiment_00/fold_0/summary.json"))


In [50]:
def extract_case_metrics(summary: Dict) -> pd.DataFrame:
    records = []
    for entry in summary["metric_per_case"]:
        case_id = Path(entry["prediction_file"]).stem  # Mets_XXX
        metrics = entry["metrics"]["1"]
        metrics["case_id"] = case_id
        records.append(metrics)
        metrics['Precision'] = metrics['TP']/(metrics['TP'] + metrics['FP'])
        metrics['Recall'] = metrics['TP']/(metrics['TP'] + metrics['FN'])
    return pd.DataFrame(records).set_index("case_id").sort_index()


case_a = extract_case_metrics(summary_a)
case_b = extract_case_metrics(summary_b)
case_a.head()

,Dice,FN,FP,IoU,TN,TP,n_pred,n_ref,precision,recall,Precision,Recall
case_id,,,,,,,,,,,,
Mets_014.nii,0.551337,1232,446,0.380583,9827691,1031,1477,2263,0.698037,0.455590,0.698037,0.455590
Mets_028.nii,0.588727,320,74,0.417160,9829724,282,356,602,0.792135,0.468439,0.792135,0.468439
Mets_036.nii,0.785903,67,419,0.647315,9829022,892,1311,959,0.680397,0.930136,0.680397,0.930136
Mets_046.nii,0.654762,56,2,0.486726,9830287,55,57,111,0.964912,0.495495,0.964912,0.495495
Mets_049.nii,0.809637,3440,1601,0.680160,9814639,10720,12321,14160,0.870059,0.757062,0.870059,0.757062


In [ ]:
print(
    f"Dice: {round(case_a['Dice'].mean(), 4)} ± {round(case_a['Dice'].std(), 4)}")
print(
    f"Precision: {round(case_a['Precision'].mean(), 4)} ± {round(case_a['Precision'].std(), 4)}")
print(
    f"Recall: {round(case_a['Recall'].mean(), 4)} ± {round(case_a['Recall'].std(), 4)}")

Dice: 0.695 ± 0.1938
Precision: 0.7691 ± 0.1977
Recall: 0.6835 ± 0.1778


In [35]:
case_a.to_csv("DiceCE.csv")
case_b.to_csv("CCDiceCE.csv")

In [51]:
def load_mask(path: Path) -> np.ndarray:
    return nib.load(path).get_fdata().astype(np.uint8)

def get_case_paths(summary: Dict) -> Dict[str, Tuple[Path, Path]]:
    out = {}
    for entry in summary["metric_per_case"]:
        case_id = Path(entry["prediction_file"]).stem
        out[case_id] = (
            Path(entry["prediction_file"]),
            Path(entry["reference_file"])
        )
    return out


paths_a = get_case_paths(summary_a)
paths_b = get_case_paths(summary_b)

In [52]:
def label_instances(mask: np.ndarray) -> Tuple[np.ndarray, int]:
    labeled, n = ndi.label(mask > 0)
    return labeled, n


def compute_iou_matrix(pred_lbl, gt_lbl, n_pred, n_gt):
    iou = np.zeros((n_gt, n_pred))
    for g in range(1, n_gt + 1):
        g_mask = gt_lbl == g
        for p in range(1, n_pred + 1):
            p_mask = pred_lbl == p
            inter = np.logical_and(g_mask, p_mask).sum()
            union = np.logical_or(g_mask, p_mask).sum()
            if union > 0:
                iou[g - 1, p - 1] = inter / union
    return iou


def compute_overlap_matrix(pred_lbl, gt_lbl, n_pred, n_gt):
    """
    Returns an (n_gt x n_pred) matrix where entry (i, j)
    is the number of overlapping voxels between GT_i and Pred_j
    """
    overlap = np.zeros((n_gt, n_pred), dtype=np.int64)

    for g in range(1, n_gt + 1):
        g_mask = gt_lbl == g
        for p in range(1, n_pred + 1):
            overlap[g - 1, p - 1] = np.logical_and(g_mask, pred_lbl == p).sum()

    return overlap

In [53]:
def match_instances_any_overlap(overlap_matrix):
    """
    One-to-one matching where a match is valid
    if overlap >= 1 voxel.
    """
    # Binary cost: maximize overlap existence
    cost = -(overlap_matrix > 0).astype(int)

    gt_idx, pred_idx = linear_sum_assignment(cost)

    matches = []
    for g, p in zip(gt_idx, pred_idx):
        if overlap_matrix[g, p] >= 1:
            matches.append((g, p))

    matched_gt = {g for g, _ in matches}
    matched_pred = {p for _, p in matches}

    return matches, matched_gt, matched_pred

In [54]:
def instance_detection_metrics(pred_path, gt_path):
    pred = load_mask(pred_path)
    gt = load_mask(gt_path)

    pred_lbl, n_pred = label_instances(pred)
    gt_lbl, n_gt = label_instances(gt)

    # Edge cases
    if n_gt == 0 and n_pred == 0:
        return {
            "TP_instances": 0,
            "FP_instances": 0,
            "FN_instances": 0,
            "instance_precision": 1.0,
            "instance_recall": 1.0,
            "instance_f1": 1.0,
        }

    if n_gt == 0:
        return {
            "TP_instances": 0,
            "FP_instances": n_pred,
            "FN_instances": 0,
            "instance_precision": 0.0,
            "instance_recall": 1.0,
            "instance_f1": 0.0,
        }

    overlap = compute_overlap_matrix(pred_lbl, gt_lbl, n_pred, n_gt)
    matches, matched_gt, matched_pred = match_instances_any_overlap(overlap)

    TP = len(matches)
    FN = n_gt - TP
    FP = n_pred - TP

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "TP_instances": TP,
        "FP_instances": FP,
        "FN_instances": FN,
        "instance_precision": precision,
        "instance_recall": recall,
        "instance_f1": f1,
    }

In [60]:
instance_f1_records = []

for case_id in sorted(paths_a.keys()):
    pred_a, gt = paths_a[case_id]
    pred_b, _ = paths_b[case_id]

    m_a = instance_detection_metrics(pred_a, gt)
    m_b = instance_detection_metrics(pred_b, gt)

    record = {"case_id": case_id}

    for k, v in m_a.items():
        record[f"{k}_DiceCE"] = v
    for k, v in m_b.items():
        record[f"{k}_CCDiceCE"] = v

    record["instance_f1_delta"] = (
        record["instance_f1_CCDiceCE"] - record["instance_f1_DiceCE"]
    )

    instance_f1_records.append(record)

instance_f1_df = (
    pd.DataFrame(instance_f1_records)
    .set_index("case_id")
    .sort_index()
)

instance_f1_df.head()

,TP_instances_DiceCE,FP_instances_DiceCE,FN_instances_DiceCE,instance_precision_DiceCE,instance_recall_DiceCE,instance_f1_DiceCE,TP_instances_CCDiceCE,FP_instances_CCDiceCE,FN_instances_CCDiceCE,instance_precision_CCDiceCE,instance_recall_CCDiceCE,instance_f1_CCDiceCE,instance_f1_delta
case_id,,,,,,,,,,,,,
Mets_014.nii,4,8,7,0.333333,0.363636,0.347826,4,14,7,0.222222,0.363636,0.275862,-0.071964
Mets_028.nii,10,1,11,0.909091,0.476190,0.625000,9,0,12,1.000000,0.428571,0.600000,-0.025000
Mets_036.nii,2,1,1,0.666667,0.666667,0.666667,2,1,1,0.666667,0.666667,0.666667,0.000000
Mets_046.nii,1,0,0,1.000000,1.000000,1.000000,1,0,0,1.000000,1.000000,1.000000,0.000000
Mets_049.nii,14,5,5,0.736842,0.736842,0.736842,14,4,5,0.777778,0.736842,0.756757,0.019915


In [62]:
instance_f1_df.to_csv('instance_df.csv')

In [61]:
def global_instance_f1(instance_df, suffix):
    TP = instance_df[f"TP_instances_{suffix}"].sum()
    FP = instance_df[f"FP_instances_{suffix}"].sum()
    FN = instance_df[f"FN_instances_{suffix}"].sum()

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return pd.Series({
        "TP_instances": TP,
        "FP_instances": FP,
        "FN_instances": FN,
        "instance_precision": precision,
        "instance_recall": recall,
        "instance_f1": f1,
    })


global_f1 = pd.DataFrame({
    "DiceCE": global_instance_f1(instance_f1_df, "DiceCE"),
    "CCDiceCE": global_instance_f1(instance_f1_df, "CCDiceCE"),
})

global_f1["delta"] = global_f1["CCDiceCE"] - global_f1["DiceCE"]
global_f1

,DiceCE,CCDiceCE,delta
TP_instances,182.000000,202.000000,20.000000
FP_instances,51.000000,101.000000,50.000000
FN_instances,109.000000,89.000000,-20.000000
instance_precision,0.781116,0.666667,-0.114449
instance_recall,0.625430,0.694158,0.068729
instance_f1,0.694656,0.680135,-0.014522
